# Задача 10. Graph Convolutional Network

Решаем задачу классификации вершин на датасете Cora из PyTorch Geometric.

Что есть в ноутбуке:

- GCN из нескольких слоёв `torch_geometric.nn.GCNConv`;
- подбор шага обучения, размера скрытого слоя и dropout по точности на валидационной выборке;
- оценка качества на тестовой выборке;
- самостоятельная реализация GCNConv через матричные операции;
- сравнение PyG GCN и собственной GCN-модели.

In [ ]:
# mypy: ignore-errors

import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and importlib.util.find_spec("torch_geometric") is None:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "torch-geometric",
        ]
    )

In [ ]:
# mypy: ignore-errors
# ruff: noqa: E402

import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch import Tensor, nn
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.utils import add_self_loops, degree

SEED = 42
DATA_DIR = Path("../data/task_10_gcn")
MAX_EPOCHS = 250
PATIENCE = 35
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Явно отключаем дорогостоящие проверки sparse-инвариантов: матрица
# смежности строится контролируемо, а PyTorch иначе печатает warning.
torch.sparse.check_sparse_tensor_invariants.disable()

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"device: {DEVICE}")

## 1. Датасет

Используем Cora: citation network, где вершины — публикации, рёбра — цитирования, признаки — bag-of-words, целевая переменная — тема публикации. Это стандартная задача классификации вершин.

In [ ]:
dataset = Planetoid(
    root=str(DATA_DIR),
    name="Cora",
    transform=NormalizeFeatures(),
)
data = dataset[0].to(DEVICE)

summary = {
    "dataset": "Cora",
    "num_graphs": len(dataset),
    "num_nodes": data.num_nodes,
    "num_edges": data.num_edges,
    "num_node_features": dataset.num_node_features,
    "num_classes": dataset.num_classes,
    "train_nodes": int(data.train_mask.sum()),
    "val_nodes": int(data.val_mask.sum()),
    "test_nodes": int(data.test_mask.sum()),
}
display(pd.DataFrame([summary]))

In [ ]:
@dataclass
class ExperimentConfig:
    model_name: str
    hidden_channels: int
    learning_rate: float
    dropout: float
    weight_decay: float = 5e-4


@dataclass
class ExperimentResult:
    config: ExperimentConfig
    best_val_accuracy: float
    test_accuracy: float
    test_macro_f1: float
    history: pd.DataFrame
    model: nn.Module


def mask_metrics(logits: Tensor, y: Tensor, mask: Tensor) -> dict[str, float]:
    y_true = y[mask].detach().cpu().numpy()
    y_pred = logits[mask].argmax(dim=1).detach().cpu().numpy()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }


def train_one_epoch(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
) -> float:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits = model(data.x, data.edge_index)
    loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return float(loss.item())


@torch.no_grad()
def evaluate_model(model: nn.Module) -> dict[str, float]:
    model.eval()
    logits = model(data.x, data.edge_index)
    val_metrics = mask_metrics(logits, data.y, data.val_mask)
    test_metrics = mask_metrics(logits, data.y, data.test_mask)
    return {
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
    }


def train_with_early_stopping(
    model: nn.Module,
    config: ExperimentConfig,
) -> ExperimentResult:
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )
    best_state = None
    best_val_accuracy = -1.0
    best_epoch = 0
    rows = []

    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(model, optimizer)
        metrics = evaluate_model(model)
        rows.append({"epoch": epoch, "train_loss": train_loss, **metrics})

        if metrics["val_accuracy"] > best_val_accuracy:
            best_val_accuracy = metrics["val_accuracy"]
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
        if epoch - best_epoch >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    final_metrics = evaluate_model(model)
    history = pd.DataFrame(rows)
    return ExperimentResult(
        config=config,
        best_val_accuracy=best_val_accuracy,
        test_accuracy=final_metrics["test_accuracy"],
        test_macro_f1=final_metrics["test_macro_f1"],
        history=history,
        model=model,
    )

## 2. GCN на слоях PyG `GCNConv`

Модель использует два слоя `GCNConv`: первый строит скрытые представления вершин, второй выдаёт логиты по классам. Гиперпараметры выбираются по точности на валидационной выборке.

In [ ]:
class PyGGCN(nn.Module):
    def __init__(
        self,
        in_channels: int,
        hidden_channels: int,
        out_channels: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x: Tensor, edge_index: Tensor) -> Tensor:
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)


def make_pyg_model(config: ExperimentConfig) -> PyGGCN:
    return PyGGCN(
        in_channels=dataset.num_node_features,
        hidden_channels=config.hidden_channels,
        out_channels=dataset.num_classes,
        dropout=config.dropout,
    )

In [ ]:
pyg_configs = [
    ExperimentConfig(
        model_name="PyG GCNConv",
        hidden_channels=hidden_channels,
        learning_rate=learning_rate,
        dropout=dropout,
    )
    for hidden_channels in [16, 32]
    for learning_rate in [0.01, 0.005, 0.001]
    for dropout in [0.3, 0.5]
]

pyg_results = []
for config in pyg_configs:
    torch.manual_seed(SEED)
    model = make_pyg_model(config)
    result = train_with_early_stopping(model, config)
    pyg_results.append(result)

pyg_search_df = pd.DataFrame(
    [
        {
            "model": result.config.model_name,
            "hidden_channels": result.config.hidden_channels,
            "learning_rate": result.config.learning_rate,
            "dropout": result.config.dropout,
            "best_val_accuracy": result.best_val_accuracy,
            "test_accuracy": result.test_accuracy,
            "test_macro_f1": result.test_macro_f1,
            "epochs": int(result.history["epoch"].max()),
        }
        for result in pyg_results
    ]
).sort_values("best_val_accuracy", ascending=False)

display(pyg_search_df.round(4))
best_pyg_result = max(pyg_results, key=lambda item: item.best_val_accuracy)
print("best PyG config:", best_pyg_result.config)

## 3. Собственная реализация GCNConv

Классический слой GCN вычисляет

`H = D_hat^{-1/2} A_hat D_hat^{-1/2} X W`,

где `A_hat = A + I`. Ниже нормализованная матрица смежности строится как разреженный тензор, а прямой проход использует `torch.sparse.mm`.

In [ ]:
def normalized_adjacency(
    edge_index: Tensor,
    num_nodes: int,
    device: torch.device,
) -> Tensor:
    edge_index_with_loops, _ = add_self_loops(
        edge_index,
        num_nodes=num_nodes,
    )
    row, col = edge_index_with_loops
    deg = degree(
        col,
        num_nodes=num_nodes,
        dtype=torch.float32,
    )
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
    edge_weight = deg_inv_sqrt[row] * deg_inv_sqrt[col]

    adjacency = torch.sparse_coo_tensor(
        edge_index_with_loops,
        edge_weight,
        size=(num_nodes, num_nodes),
        device=device,
        check_invariants=False,
    )
    return adjacency.coalesce()


class MatrixGCNConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.linear = nn.Linear(in_channels, out_channels, bias=False)
        self.bias = nn.Parameter(torch.zeros(out_channels))

    def forward(self, x: Tensor, adjacency: Tensor) -> Tensor:
        support = self.linear(x)
        out = torch.sparse.mm(adjacency, support)
        return out + self.bias


class MatrixGCN(nn.Module):
    def __init__(
        self,
        in_channels: int,
        hidden_channels: int,
        out_channels: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.conv1 = MatrixGCNConv(in_channels, hidden_channels)
        self.conv2 = MatrixGCNConv(hidden_channels, out_channels)
        self.dropout = dropout
        self.register_buffer("adjacency", torch.empty(0))

    def set_graph(self, edge_index: Tensor, num_nodes: int) -> None:
        self.adjacency = normalized_adjacency(edge_index, num_nodes, DEVICE)

    def forward(self, x: Tensor, edge_index: Tensor) -> Tensor:
        if self.adjacency.numel() == 0:
            self.set_graph(edge_index, x.shape[0])
        x = self.conv1(x, self.adjacency)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, self.adjacency)


def make_matrix_model(config: ExperimentConfig) -> MatrixGCN:
    model = MatrixGCN(
        in_channels=dataset.num_node_features,
        hidden_channels=config.hidden_channels,
        out_channels=dataset.num_classes,
        dropout=config.dropout,
    )
    model.set_graph(data.edge_index, data.num_nodes)
    return model

In [ ]:
matrix_configs = [
    ExperimentConfig(
        model_name="Matrix GCNConv",
        hidden_channels=hidden_channels,
        learning_rate=learning_rate,
        dropout=dropout,
    )
    for hidden_channels in [16, 32]
    for learning_rate in [0.01, 0.005, 0.001]
    for dropout in [0.3, 0.5]
]

matrix_results = []
for config in matrix_configs:
    torch.manual_seed(SEED)
    model = make_matrix_model(config)
    result = train_with_early_stopping(model, config)
    matrix_results.append(result)

matrix_search_df = pd.DataFrame(
    [
        {
            "model": result.config.model_name,
            "hidden_channels": result.config.hidden_channels,
            "learning_rate": result.config.learning_rate,
            "dropout": result.config.dropout,
            "best_val_accuracy": result.best_val_accuracy,
            "test_accuracy": result.test_accuracy,
            "test_macro_f1": result.test_macro_f1,
            "epochs": int(result.history["epoch"].max()),
        }
        for result in matrix_results
    ]
).sort_values("best_val_accuracy", ascending=False)

display(matrix_search_df.round(4))
best_matrix_result = max(matrix_results, key=lambda item: item.best_val_accuracy)
print("best Matrix GCN config:", best_matrix_result.config)

## 4. Сравнение результатов

Сравним лучшие модели по точности на валидационной выборке и оценим их на тестовой выборке. Дополнительно построим кривые обучения для выбранных запусков.

In [ ]:
def result_to_row(result: ExperimentResult) -> dict[str, float | int | str]:
    return {
        "model": result.config.model_name,
        "hidden_channels": result.config.hidden_channels,
        "learning_rate": result.config.learning_rate,
        "dropout": result.config.dropout,
        "best_val_accuracy": result.best_val_accuracy,
        "test_accuracy": result.test_accuracy,
        "test_macro_f1": result.test_macro_f1,
        "epochs": int(result.history["epoch"].max()),
    }


comparison_df = pd.DataFrame(
    [
        result_to_row(best_pyg_result),
        result_to_row(best_matrix_result),
    ]
).sort_values("test_accuracy", ascending=False)

display(comparison_df.round(4))

In [ ]:
def plot_history(result: ExperimentResult) -> None:
    history = result.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["epoch"], history["train_loss"])
    axes[0].set_title(f"{result.config.model_name}: train loss")
    axes[0].set_xlabel("epoch")

    axes[1].plot(history["epoch"], history["val_accuracy"], label="val")
    axes[1].plot(history["epoch"], history["test_accuracy"], label="test")
    axes[1].set_title(f"{result.config.model_name}: accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    plt.show()


plot_history(best_pyg_result)
plot_history(best_matrix_result)

In [ ]:
@torch.no_grad()
def print_test_report(result: ExperimentResult) -> None:
    result.model.eval()
    logits = result.model(data.x, data.edge_index)
    y_true = data.y[data.test_mask].detach().cpu().numpy()
    y_pred = logits[data.test_mask].argmax(dim=1).detach().cpu().numpy()
    print(result.config.model_name)
    print(classification_report(y_true, y_pred, digits=4))


print_test_report(best_pyg_result)
print_test_report(best_matrix_result)

In [ ]:
best_row = comparison_df.iloc[0]
accuracy_delta = best_matrix_result.test_accuracy - best_pyg_result.test_accuracy
f1_delta = best_matrix_result.test_macro_f1 - best_pyg_result.test_macro_f1

print("Выводы")
print("======")
print(
    "1. Для классификации вершин выбран Cora: один граф цитирований "
    "с фиксированными масками обучающей, валидационной и тестовой выборок."
)
print(
    "2. PyG-модель использует два слоя GCNConv и подбирает размер "
    "скрытого слоя, шаг обучения и dropout по точности на валидации."
)
print(
    "3. Собственный MatrixGCNConv реализует ту же нормализацию "
    "D^{-1/2}(A+I)D^{-1/2} и умножение на разреженную матрицу."
)
print(
    "4. Собственный слой изменил точность на тестовой выборке "
    f"на {accuracy_delta:+.4f} и macro-F1 на {f1_delta:+.4f} "
    "относительно PyG GCNConv."
)
print(
    "5. Лучший запуск: "
    f"{best_row['model']} с test_accuracy={best_row['test_accuracy']:.4f} "
    f"и test_macro_f1={best_row['test_macro_f1']:.4f}."
)